# 3dgs-lab-colab: 速さ重視の3D Gaussian Splatting (CUDA/gsplat)

ローカル版(`3dgs-lab/`, Mac + Brush + COLMAP CPU)と同じ入力(動画または画像)から、
**Colab の NVIDIA GPU(CUDA)** を使って [gsplat](https://github.com/nerfstudio-project/gsplat) で学習し、
`.ply` だけをダウンロードして持ち帰る構成です。

- ビューアはローカルの `3dgs-lab/viewer/index.html` をそのまま使えます(`.ply` を読むだけなので学習場所は問いません)。
- **メニュー > ランタイム > ランタイムのタイプを変更 で GPU(T4 など)を選択してから実行してください。**
- 無料枠のT4はセッション時間制限・切断があります。Colab Pro/Pro+ならより高速なGPU(A100等)が使え、ステップ数や解像度をさらに上げられます。

**注記**: このNotebookは gsplat の公式ソース(README / `examples/simple_trainer.py`)を実際に読んで作成していますが、
Colab の GPU ランタイム上での実行はまだ行っていません(Claude Code はColabの実行環境に直接アクセスできないため)。
初回実行でpip installのエラー等が出た場合は、内容を教えてもらえれば修正します。

## 0. GPUの確認

In [ ]:
!nvidia-smi

## 1. セットアップ
COLMAP・ffmpeg(apt)と gsplat(pip)を導入します。初回は数分かかります。

In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap ffmpeg
!colmap -h | head -1
!ffmpeg -version | head -1

In [ ]:
import os, shutil

# mainブランチはセンサー/LiDAR/推論用など大量のCUDAコードが増えており、
# フルビルド(pip install git+...)がColabのT4上で1時間以上かかり実用にならなかった。
# PyPIに公開されている gsplat==1.5.3(JITビルド、mainよりずっと小さい)と、
# それに対応する v1.5.3 タグの examples/ を使うことでビルド量を最小限にする。
shutil.rmtree("/content/gsplat_repo", ignore_errors=True)
!git clone -q --branch v1.5.3 --depth 1 https://github.com/nerfstudio-project/gsplat.git /content/gsplat_repo

# Colab既定のtorchはCUDA 13.0ビルドだが、VM上のnvccは12.8のみ(nvidia-cuda-nvcc-cu13は
# PyPI上に中身のないプレースホルダーで使えなかった)。system側のnvcc(12.8)に合わせて
# torch自体をCUDA 12.8ビルドに入れ替える。
!bash -o pipefail -c 'pip install --force-reinstall --index-url https://download.pytorch.org/whl/cu128 torch 2>&1 | tail -n 100'
assert _exit_code == 0, f"torchの入れ替えに失敗しました(exit code {_exit_code})"

import torch
print('torch', torch.__version__, 'torch cuda build:', torch.version.cuda, 'cuda available:', torch.cuda.is_available())
assert torch.version.cuda is not None and torch.version.cuda.startswith("12.8"), \
    f"torchがまだCUDA12.8ビルドになっていません(現在: {torch.version.cuda})"

!pip install -q gsplat==1.5.3

# v1.5.3のdatasets/colmap.pyは `from pycolmap import SceneManager` という、
# 公式pycolmap(pip install pycolmap)には無いAPIを使う非公式フォークを要求する
# (mainブランチのcolmap.pyは公式pycolmapに書き換わっているが、v1.5.3タグは旧式のまま)。
!pip install -q "git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e"

# v1.5.3のexamples/simple_trainer.pyはfused_ssimをトップレベルで必須importしている
# (mainブランチはgsplat.losses配下に内製化されており不要)。
# --no-build-isolation は既に入れ替えたtorch(12.8)を隔離ビルド環境に取られず使わせるため。
!pip install -q ninja
!bash -o pipefail -c 'pip install -v --no-build-isolation "git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5" 2>&1 | tail -n 100'
assert _exit_code == 0, f"fused-ssimのインストールに失敗しました(exit code {_exit_code})。上のログの実際のコンパイルエラー行を確認してください。"

!pip install -q opencv-python-headless "imageio[ffmpeg]" tqdm \
    "tyro>=0.8.8,!=1.0.9,!=1.0.10" pyyaml matplotlib scikit-learn \
    torchmetrics lpips piexif tensorboard viser splines
!pip install -q "git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787"

import torch, gsplat
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
print('gsplat', gsplat.__version__)

## 2. 設定
ローカル版の `--preset` に相当する設定です。速さ重視のデフォルトになっています。

In [ ]:
SCENE_NAME = "myscene"  #@param {type:"string"}
INPUT_TYPE = "video"  #@param ["video", "images"]
FRAMES_TARGET = 200  #@param {type:"integer"}
LONG_EDGE = 1600  #@param {type:"integer"}
MAX_STEPS = 15000  #@param {type:"integer"}
CAP_MAX_SPLATS = 1500000  #@param {type:"integer"}
SH_DEGREE = 3  #@param {type:"integer"}

import os
ROOT = f"/content/{SCENE_NAME}"
IMAGES_DIR = f"{ROOT}/images"
SPARSE_DIR = f"{ROOT}/sparse"
RESULT_DIR = f"/content/results/{SCENE_NAME}"
os.makedirs(IMAGES_DIR, exist_ok=True)
print("scene root:", ROOT)

## 3. 入力のアップロード
`INPUT_TYPE = "video"` なら動画ファイルを1つ、`"images"` なら画像複数 or 画像を固めたzipをアップロードしてください。

In [ ]:
from google.colab import files
import shutil, zipfile, glob

uploaded = files.upload()
uploaded_paths = []
for name, data in uploaded.items():
    dst = f"/content/upload_{name}"
    with open(dst, "wb") as f:
        f.write(data)
    uploaded_paths.append(dst)

VIDEO_PATH = None
RAW_IMAGES_DIR = f"{ROOT}/raw_images"
os.makedirs(RAW_IMAGES_DIR, exist_ok=True)

if INPUT_TYPE == "video":
    assert len(uploaded_paths) >= 1, "動画ファイルを1つアップロードしてください"
    VIDEO_PATH = uploaded_paths[0]
    print("video:", VIDEO_PATH)
else:
    for p in uploaded_paths:
        if p.lower().endswith(".zip"):
            with zipfile.ZipFile(p) as zf:
                zf.extractall(RAW_IMAGES_DIR)
        else:
            shutil.copy(p, RAW_IMAGES_DIR)
    found = glob.glob(f"{RAW_IMAGES_DIR}/**/*.*", recursive=True)
    print(f"{len(found)} 個のファイルを検出")

## 4. フレーム抽出 / 画像整形
ローカル版の `extract` ステージと同じロジック(目標フレーム数へのfpsサンプリング、長辺リサイズ)です。

In [ ]:
import subprocess

def scale_filter_expr(long_edge):
    return (
        f"scale=w='if(gt(iw,ih),min(iw,{long_edge}),-2)':"
        f"h='if(gt(iw,ih),-2,min(ih,{long_edge}))'"
    )

if INPUT_TYPE == "video":
    dur = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", VIDEO_PATH],
        capture_output=True, text=True,
    )
    duration = float(dur.stdout.strip())
    fps = max(FRAMES_TARGET / duration, 0.1)
    print(f"duration={duration:.1f}s fps={fps:.4f}")
    vf = f"fps={fps:.6f}"
    if LONG_EDGE > 0:
        vf += "," + scale_filter_expr(LONG_EDGE)
    subprocess.run(
        ["ffmpeg", "-y", "-i", VIDEO_PATH, "-vf", vf, "-q:v", "2",
         f"{IMAGES_DIR}/frame_%05d.jpg"],
        check=True,
    )
else:
    from PIL import Image, ImageOps
    src_files = sorted(glob.glob(f"{RAW_IMAGES_DIR}/**/*.*", recursive=True))
    src_files = [p for p in src_files if p.lower().rsplit(".", 1)[-1] in ("jpg", "jpeg", "png", "bmp", "tif", "tiff")]
    for i, src in enumerate(src_files, start=1):
        with Image.open(src) as img:
            img = ImageOps.exif_transpose(img)
            if img.mode != "RGB":
                img = img.convert("RGB")
            if LONG_EDGE > 0:
                w, h = img.size
                scale = min(1.0, LONG_EDGE / max(w, h))
                if scale < 1.0:
                    img = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.LANCZOS)
            img.save(f"{IMAGES_DIR}/frame_{i:05d}.jpg", quality=95)

n_images = len(glob.glob(f"{IMAGES_DIR}/*.jpg"))
print(f"{n_images} 枚の画像を用意しました -> {IMAGES_DIR}")
assert n_images > 0, "画像を1枚も取得できませんでした"

## 5. COLMAP (SfM)
ローカル版と同じ設定(`OPENCV`カメラモデル、動画なら`sequential_matcher`/写真なら`exhaustive_matcher`)。
Colabの apt 版 COLMAP は基本CPUビルドですが、SfM自体は全体の数%程度なのでボトルネックにはなりません。

In [ ]:
import os, subprocess, shutil

# Colabのapt版COLMAPはGUI(Qt)付きビルドで、表示(ディスプレイ)が無い環境だと
# feature_extractor/matcherがQApplication初期化でクラッシュすることがある
# (qt.qpa.xcb: could not connect to display)。offscreenプラットフォームを使わせて回避する。
os.environ["QT_QPA_PLATFORM"] = "offscreen"

DB_PATH = f"{ROOT}/colmap.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
if os.path.exists(SPARSE_DIR):
    shutil.rmtree(SPARSE_DIR)
os.makedirs(SPARSE_DIR, exist_ok=True)

def gpu_flag(subcommand, on_name, off_name):
    """COLMAPのバージョンによってuse_gpuオプション名が異なる(SiftExtraction.* vs FeatureExtraction.*等)ため、
    -h の出力を見てこの環境に実在する方を採用する。"""
    help_text = subprocess.run(["colmap", subcommand, "-h"], capture_output=True, text=True).stdout
    return on_name if on_name in help_text else off_name

extract_gpu_flag = gpu_flag("feature_extractor", "FeatureExtraction.use_gpu", "SiftExtraction.use_gpu")

!colmap feature_extractor \
  --database_path {DB_PATH} \
  --image_path {IMAGES_DIR} \
  --ImageReader.single_camera 1 \
  --ImageReader.camera_model OPENCV \
  --{extract_gpu_flag} 0

matcher = "sequential_matcher" if INPUT_TYPE == "video" else "exhaustive_matcher"
match_gpu_flag = gpu_flag(matcher, "FeatureMatching.use_gpu", "SiftMatching.use_gpu")
!colmap {matcher} --database_path {DB_PATH} --{match_gpu_flag} 0

!colmap mapper \
  --database_path {DB_PATH} \
  --image_path {IMAGES_DIR} \
  --output_path {SPARSE_DIR}

!colmap model_analyzer --path {SPARSE_DIR}/0

上の出力の `Registered images` が入力枚数よりだいぶ少ない場合、撮影(周回不足・ブレなど)を見直してください。

## 6. 学習 (gsplat, CUDA / MCMC戦略)
`mcmc` サブコマンドがBrushの `--max-splats` に相当する `cap_max` を持つ戦略です。

In [ ]:
import time
%cd /content/gsplat_repo/examples

# strategy配下のフィールドだけはtyroがCLIフラグをハイフン区切りにする
# (公式 examples/benchmarks/mcmc.sh でも --strategy.cap-max と表記されている。
# トップレベルのフィールド(--data_dir 等)はアンダースコアのままでよい)。
# --strategy.cap_max のようにアンダースコアで渡すと引数解析エラーで
# simple_trainer.pyが即終了し、ply/ディレクトリが作られない。
t0 = time.time()
!python simple_trainer.py mcmc \
  --data_dir {ROOT} \
  --data_factor 1 \
  --result_dir {RESULT_DIR} \
  --max_steps {MAX_STEPS} \
  --strategy.cap-max {CAP_MAX_SPLATS} \
  --sh_degree {SH_DEGREE} \
  --save_ply \
  --disable_viewer \
  --disable_video
print(f"elapsed: {time.time() - t0:.1f}s")
assert _exit_code == 0, (
    f"学習コマンドがエラー終了しました(exit code {_exit_code})。"
    "上のセル出力に表示されているエラーメッセージを確認してください。"
)

COLMAPのデータ配置(`{ROOT}/images` + `{ROOT}/sparse/0/...`)は Mip-NeRF360 標準レイアウトで、
gsplatの`datasets/colmap.py`が期待する形式そのままです(ローカル版`splat.py`の出力と同じ考え方)。

## 7. .ply のダウンロード

In [ ]:
import glob, os
from google.colab import files

ply_files = sorted(glob.glob(f"{RESULT_DIR}/ply/point_cloud_*.ply"),
                    key=lambda p: int(p.rsplit("_", 1)[-1].split(".")[0]))
assert ply_files, f"plyが見つかりません: {RESULT_DIR}/ply/"
final_ply = ply_files[-1]
print("final ply:", final_ply, f"({os.path.getsize(final_ply)/1e6:.1f} MB)")

out_name = f"{SCENE_NAME}.ply"
shutil_copy_path = f"/content/{out_name}"
import shutil
shutil.copy(final_ply, shutil_copy_path)
files.download(shutil_copy_path)

## 8. ローカルで見る

ダウンロードした `.ply` は、ローカルの `3dgs-lab/viewer/index.html` にそのままドラッグ&ドロップして表示できます。

または `3dgs-lab/output/<シーン名>/<シーン名>.ply` として配置し、

```bash
cd 3dgs-lab
python3 splat.py --name <シーン名> --only view
```

を実行してください(学習はしていないので一瞬でビューアが開きます)。